# Simplified Local Hybrid RAG (All-Qdrant)

This notebook implements a production-grade, fully local RAG pipeline.
- **Vector Store:** Qdrant (handles both Dense and Sparse vectors, plus metadata).
- **Dense Embeddings:** Nomic (v1.5 or v2) via LM Studio.
- **Sparse Embeddings:** BM25 via FastEmbed (local CPU).
- **Reranking:** ms-marco-MiniLM (local CPU).
- **Generation:** Gemma 4 via LM Studio.

In [ ]:
!pip install -qU qdrant-client fastembed sentence-transformers langchain-openai

### 1. Initialize Local Models

In [ ]:
from fastembed import SparseTextEmbedding
from sentence_transformers import CrossEncoder
from langchain_openai import OpenAIEmbeddings, ChatOpenAI

# 1. Sparse Embedding Model (BM25)
# Downloads a tiny vocabulary mapping to compute word frequencies locally
sparse_model = SparseTextEmbedding(model_name="Qdrant/bm25")

# 2. Cross-Encoder for Reranking
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

# 3. Dense Embedding Model (LM Studio - Nomic)
dense_embeddings = OpenAIEmbeddings(
    base_url="http://localhost:1234/v1",
    api_key="lm-studio",
    model="text-embedding-nomic" # Replace with your loaded Nomic model name
)

# 4. Generation Model (LM Studio - Gemma 4)
llm = ChatOpenAI(
    base_url="http://localhost:1234/v1",
    api_key="lm-studio",
    model="gemma-4", # Replace with your loaded Gemma model name
    temperature=0
)

### 2. Configure Qdrant Collection

In [ ]:
from qdrant_client import QdrantClient, models

# Connect to local Qdrant instance
client = QdrantClient(url="http://localhost:6333")
COLLECTION_NAME = "hybrid_documents"

# Recreate the collection with two vector spaces: 'dense' and 'sparse'
if client.collection_exists(COLLECTION_NAME):
    client.delete_collection(COLLECTION_NAME)

client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config={
        "dense": models.VectorParams(
            size=768, # Nomic's default dimension size
            distance=models.Distance.COSINE
        )
    },
    sparse_vectors_config={
        "sparse": models.SparseVectorParams(
            modifier=models.Modifier.IDF # Let Qdrant handle Inverse Document Frequency weighting
        )
    }
)
print(f"Collection '{COLLECTION_NAME}' created successfully.")

### 3. Ingest Data (Dense + Sparse + Payload)

In [ ]:
# Sample data
documents = [
    "The Apollo 11 mission landed humans on the Moon in 1969. Neil Armstrong was the first to step on the surface.",
    "Project Artemis is NASA's current program to return humans to the Moon and eventually send them to Mars.",
    "The Saturn V was the super heavy-lift launch vehicle used by NASA between 1967 and 1973.",
    "SpaceX's Starship is a fully reusable spacecraft designed to carry both crew and cargo to Earth orbit, the Moon, and Mars.",
    "Hybrid search combines semantic similarity with keyword matching to retrieve the most relevant documents."
]

print("Generating embeddings...")
# Generate Dense Vectors via LM Studio
dense_vectors = dense_embeddings.embed_documents(documents)

# Generate Sparse Vectors via FastEmbed
sparse_vectors = list(sparse_model.embed(documents))

# Construct Qdrant Points
points = []
for i, doc in enumerate(documents):
    points.append(
        models.PointStruct(
            id=i + 1,
            vector={
                "dense": dense_vectors[i],
                "sparse": models.SparseVector(
                    indices=sparse_vectors[i].indices.tolist(),
                    values=sparse_vectors[i].values.tolist()
                )
            },
            payload={"chunk_text": doc} # Store the raw text directly in Qdrant
        )
    )

client.upsert(collection_name=COLLECTION_NAME, points=points)
print(f"Successfully ingested {len(points)} documents into Qdrant.")

### 4. The Retrieval Engine (Qdrant RRF + Cross-Encoder)

In [ ]:
def retrieve_and_rerank(query: str, top_k_initial: int = 10, top_k_final: int = 3):
    # 1. Embed the query (Dense + Sparse)
    dense_query = dense_embeddings.embed_query(query)
    sparse_query_obj = list(sparse_model.embed([query]))[0]
    sparse_query = models.SparseVector(
        indices=sparse_query_obj.indices.tolist(),
        values=sparse_query_obj.values.tolist()
    )

    # 2. Hybrid Search with Native RRF in Qdrant
    results = client.query_points(
        collection_name=COLLECTION_NAME,
        prefetch=[
            models.Prefetch(query=dense_query, using="dense", limit=top_k_initial),
            models.Prefetch(query=sparse_query, using="sparse", limit=top_k_initial),
        ],
        query=models.FusionQuery(fusion=models.Fusion.RRF),
        limit=top_k_initial,
        with_payload=True
    )
    
    # Extract text payloads from Qdrant results
    initial_docs = [hit.payload["chunk_text"] for hit in results.points]
    if not initial_docs:
        return []

    # 3. Cross-Encoder Reranking
    rerank_pairs = [[query, doc] for doc in initial_docs]
    ce_scores = cross_encoder.predict(rerank_pairs)

    # Zip docs and scores, sort descending
    scored_docs = list(zip(initial_docs, ce_scores))
    scored_docs.sort(key=lambda x: x[1], reverse=True)

    # Return only the top final text chunks
    return [doc for doc, _ in scored_docs[:top_k_final]]

### 5. Full RAG Pipeline

In [ ]:
from langchain.prompts import ChatPromptTemplate

prompt_template = ChatPromptTemplate.from_template(
    """Answer the question based ONLY on the following context.\n\n"
    "Context:\n{context}\n\n"
    "Question: {question}"""
)

def generate_answer(question: str):
    print("Searching and reranking...")
    best_chunks = retrieve_and_rerank(question)
    
    if not best_chunks:
        return "No relevant context found."
        
    context_string = "\n\n---\n\n".join(best_chunks)
    
    print("Generating answer with Gemma 4...")
    chain = prompt_template | llm
    response = chain.invoke({
        "context": context_string,
        "question": question
    })
    
    return response.content

# Execute a test query
test_query = "Which vehicle took astronauts to the Moon, and what are its dates of operation?"
answer = generate_answer(test_query)

print(f"\nQuestion: {test_query}\n")
print(f"Answer:\n{answer}")